# Bài tập lớn 1 — SAT Encoding for Logic Puzzles
## Sudoku Solver using SAT / PySAT

**Course:** Các vấn đề hiện đại KHMT  
**Semester:** HKI 2026–2027  
**Topic:** SAT Encoding for Logic Puzzles — Sudoku Solver

This notebook is a research/implementation template designed around the assignment requirements:
- implement a Sudoku solver using SAT/PySAT;
- investigate ALO/AMO encodings such as Binary/Pairwise, Commander, Product and Sequential Counter;
- compare SAT solvers;
- compare SAT with exact approaches such as CP, CP-SAT, ILP/MIP, CPLEX and Gurobi where available;
- run experiments on instances of different difficulty/size;
- report correctness, variables, clauses/constraints, runtime and scalability;
- provide reproducible code and GitHub results.

**Important:** this notebook is a structure/template. Replace TODO sections with your actual implementation and measured results. Do not fabricate unavailable solver results.


In [1]:
from math import isqrt
from pysat.solvers import Glucose3
import importlib
import sudoku_problems  
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sudoku_problems import generate_puzzle
from sudoku_problem36x36 import generate_puzzle36x36
importlib.reload(sudoku_problems)
from docplex.cp.model import CpoModel
from ortools.sat.python import cp_model
import time
import gurobipy as gp
from gurobipy import GRB
import time
from docplex.mp.model import Model
import random




[[0, 22, 32, 36, 23, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 21, 0, 0, 31, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 11], [30, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 13, 0, 0, 0, 0, 0, 35, 17, 36, 0, 0, 0, 0, 0, 0, 31, 19, 21, 0, 0], [0, 35, 0, 0, 0, 9, 8, 0, 0, 0, 5, 0, 0, 0, 0, 0, 34, 7, 0, 0, 18, 0, 2, 13, 0, 10, 0, 0, 0, 0, 0, 0, 0, 15, 0, 0], [4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 19, 0, 0, 0, 32, 0, 0, 0, 0, 0, 0, 0, 0, 12, 16, 0, 0, 29, 0, 0, 0], [0, 0, 0, 3, 19, 10, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 24, 0, 0, 0, 0, 1, 35, 0, 0, 0, 0, 0, 2, 0, 0], [0, 16, 33, 0, 0, 0, 25, 19, 31, 21, 0, 3, 0, 1, 0, 0, 17, 0, 5, 0, 29, 0, 6, 0, 0, 0, 0, 2, 14, 0, 0, 22, 0, 0, 32, 36], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 19, 0, 0, 0, 21, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 24, 0, 0, 0, 0, 8, 20, 0, 0], [0, 12, 0, 0, 0, 0, 31, 25, 19, 3, 0, 0, 0, 27, 17, 0, 0, 0, 0, 20, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 23, 0, 0, 0, 5, 20, 8, 0, 0, 29, 15, 0, 0

# 1. Research Questions

### Main question
How does the choice of SAT encoding affect Sudoku solving performance, and how does SAT compare with other exact solving approaches?

### Sub-questions
1. How can Sudoku be represented as Boolean SAT?
2. What are ALO and AMO constraints?
3. How do Pairwise, Commander, Product and Sequential Counter encodings differ?
4. How do they affect variables, clauses and solving time?
5. Which SAT solver performs best on the selected benchmark?
6. How does SAT compare with CP-SAT, CP, ILP/MIP, CPLEX and Gurobi?
7. How does performance scale with Sudoku size/difficulty?


# 2. Mathematical Formulation

For an order-$n$ Sudoku, define

$$
x_{r,c,v} = 1
$$

if cell $(r, c)$ contains value $v$.

There are

$$
n^3
$$

primary Boolean variables. For standard $9 \times 9$ Sudoku:

$$
9^3 = 729.
$$

Every cell, every row/value pair, every column/value pair, and every box/value pair must satisfy an **Exactly-One** constraint.

Exactly-One is:

$$
\operatorname{ALO}(X) \land \operatorname{AMO}(X)
$$

where:

$$
\operatorname{ALO}(x_1, \ldots, x_n) = x_1 \lor \cdots \lor x_n
$$

and AMO prohibits every pair from being simultaneously true.


# 3. Sudoku SAT Model

## Cell constraints

For every cell $(r, c)$:

$$
x_{r,c,1} \lor \cdots \lor x_{r,c,n}
$$

and an AMO constraint over those $n$ variables.

## Row constraints

For every row $r$ and value $v$:

$$
\operatorname{ExactlyOne}(x_{r,1,v}, \ldots, x_{r,n,v})
$$

## Column constraints

For every column $c$ and value $v$:

$$
\operatorname{ExactlyOne}(x_{1,c,v}, \ldots, x_{n,c,v})
$$

## Box constraints

For every box $B$ and value $v$:

$$
\operatorname{ExactlyOne}(x_{r,c,v} : (r, c) \in B)
$$

## Given clues

If the puzzle contains $S_{r,c} = v$, add the unit clause:

$$
(x_{r,c,v})
$$


# 4. ALO / AMO Encoding

This is the core experimental part.

For literals $X = \{x_1, \ldots, x_n\}$:

### ALO — At Least One

$$
(x_1 \lor x_2 \lor \cdots \lor x_n)
$$

### AMO — At Most One

No two literals may be true simultaneously.

### Exactly-One

$$
\operatorname{ALO}(X) \land \operatorname{AMO}(X)
$$

The project should compare at least:

1. Pairwise / Binary AMO
2. Commander AMO
3. Product AMO
4. Sequential Counter AMO

Keep the Sudoku model identical and change only the AMO encoding so the comparison is fair.


In [2]:
from itertools import combinations, product
from math import ceil, sqrt


def alo(lits):
    return [list(lits)]


def pairwise_amo(lits):
    return [[-a, -b] for a, b in combinations(lits, 2)]


def pairwise_exactly_one(lits):
    return alo(lits) + pairwise_amo(lits)


def _start_var(lits, next_var):
    return next_var if next_var is not None else max(abs(l) for l in lits) + 1


def sequential_amo(lits, next_var=None):
    n = len(lits)
    if n <= 1:
        return [], next_var
    next_var = _start_var(lits, next_var)

    s = list(range(next_var, next_var + n - 1))
    clauses = [[-lits[0], s[0]]]
    for i in range(1, n - 1):
        clauses.append([-lits[i], s[i]])
        clauses.append([-s[i - 1], s[i]])
        clauses.append([-lits[i], -s[i - 1]])
    clauses.append([-lits[n - 1], -s[n - 2]])
    return clauses, next_var + n - 1


def commander_amo(lits, next_var=None, group_size=3):
    """
    Commander Encoding for AMO (Klieber & Kwon).

    Split literals into groups of `group_size`. Inside a group use pairwise AMO.
    Each group gets a commander variable c with (l -> c) for every l in the group.
    Then recursively enforce AMO over the commanders.
    """
    assert group_size >= 2, "group_size must be >= 2 for the recursion to terminate"
    n = len(lits)
    if n <= 1:
        return [], next_var
    next_var = _start_var(lits, next_var)

    if n <= group_size:
        return pairwise_amo(lits), next_var

    clauses, commanders = [], []
    for i in range(0, n, group_size):
        group = lits[i:i + group_size]
        if len(group) == 1:
            commanders.append(group[0])      # singleton group: literal is its own commander
            continue
        clauses += pairwise_amo(group)
        c = next_var
        next_var += 1
        clauses += [[-l, c] for l in group]  # l -> c (enough for AMO)
        commanders.append(c)

    sub, next_var = commander_amo(commanders, next_var, group_size)
    return clauses + sub, next_var


def product_amo(lits, next_var=None, base=6):
    """
    Product Encoding for AMO (Chen, 2010).

    Place literals in a p x q grid (p = ceil(sqrt(n)), q = ceil(n/p)).
    Row variables u_i and column variables v_j; literal at (i, j) implies u_i and v_j.
    Two distinct literals differ in a row or a column, so AMO over u and over v
    (applied recursively) forbids them being true together.
    """
    n = len(lits)
    if n <= 1:
        return [], next_var
    next_var = _start_var(lits, next_var)

    if n <= base:
        return pairwise_amo(lits), next_var

    p = ceil(sqrt(n))
    q = ceil(n / p)

    u = list(range(next_var, next_var + p))
    next_var += p
    v = list(range(next_var, next_var + q))
    next_var += q

    clauses = []
    for idx, x in enumerate(lits):
        i, j = divmod(idx, q)
        clauses.append([-x, u[i]])
        clauses.append([-x, v[j]])

    sub_u, next_var = product_amo(u, next_var, base)
    sub_v, next_var = product_amo(v, next_var, base)
    return clauses + sub_u + sub_v, next_var

from math import ceil, log2

def binary_amo(lits, next_var=None):
    """
    Binary (bitwise) Encoding for AMO (Frisch & Giannaros).

    Give literal i the index i in binary. Introduce m = ceil(log2 n) auxiliary
    bit variables b_0..b_{m-1}. If x_i is true, the b's must spell out i:
        x_i -> b_k      if bit k of i is 1
        x_i -> not b_k  if bit k of i is 0
    Two different literals have different indices, so some bit conflicts
    and they can't both be true.
    """
    n = len(lits)
    if n <= 1:
        return [], next_var
    next_var = _start_var(lits, next_var)

    m = ceil(log2(n))
    bits = list(range(next_var, next_var + m))

    clauses = []
    for i, x in enumerate(lits):
        for k in range(m):
            clauses.append([-x, bits[k] if (i >> k) & 1 else -bits[k]])

    return clauses, next_var + m


def exactly_one(lits, amo_fn=pairwise_amo, next_var=None, **kw):
    """ALO + any AMO encoding. Returns (clauses, next_var)."""
    if amo_fn is pairwise_amo:
        return alo(lits) + pairwise_amo(lits), next_var
    amo, next_var = amo_fn(lits, next_var, **kw)
    return alo(lits) + amo, next_var


# ---------- brute-force correctness check ----------
def check_amo(amo_fn, n, **kw):
    lits = list(range(1, n + 1))
    clauses, nv = amo_fn(lits, n + 1, **kw)
    aux = list(range(n + 1, nv)) if nv else []

    def sat(assign):
        return all(any(assign[abs(l)] == (l > 0) for l in c) for c in clauses)

    for bits in product([False, True], repeat=n):
        base = {i + 1: b for i, b in enumerate(bits)}
        ok = any(sat({**base, **dict(zip(aux, ab))})
                 for ab in product([False, True], repeat=len(aux)))
        assert ok == (sum(bits) <= 1), (amo_fn.__name__, n, bits)


if __name__ == "__main__":
    for n in range(2, 10):
        check_amo(sequential_amo, n)
        check_amo(commander_amo, n, group_size=3)
        check_amo(product_amo, n, base=3)
    print("all AMO encodings OK")

all AMO encodings OK


# 5. Encoding Comparison — What to Measure

For each encoding report:

- primary variables;
- auxiliary variables;
- total variables;
- total clauses;
- number of binary clauses;
- number of non-binary clauses;
- encoding time;
- solving time;
- total time;
- solved/timeout count;
- correctness.

Do not assume that fewer clauses means faster solving. SAT performance also depends on propagation structure and solver heuristics.


In [3]:
from dataclasses import dataclass

@dataclass
class EncodingStats:
    name: str
    variables: int
    auxiliary_variables: int
    clauses: int


# 6. Variable Numbering

Use a deterministic mapping from $(r, c, v)$ to a positive SAT variable ID.

With zero-based indices:

$$
\operatorname{id}(r, c, v) = ((r \cdot n) + c) \cdot n + v + 1
$$

The first $n^3$ IDs are primary Sudoku variables. Auxiliary variables should start after them.


In [4]:
def var_id(r, c, v, n):
    return (r * n + c) * n + v + 1

def decode_var(var, n):
    var -= 1
    v = var % n
    var //= n
    c = var % n
    r = var // n
    return r, c, v


# 7. Sudoku CNF Generator

Recommended architecture:

```text
SudokuCNF
 ├── add_cell_constraints()
 ├── add_row_constraints()
 ├── add_column_constraints()
 ├── add_box_constraints()
 └── add_clues()
```

Every Exactly-One constraint should call the selected ALO/AMO encoder.

The Sudoku generator must not depend on which encoding is being tested.


In [5]:
from math import ceil, log2, sqrt, isqrt


# ---------- helpers ----------
def var_id(r, c, v, n):
    """Variable for 'cell (r,c) has value v' (r, c, v are 0-based). IDs start at 1."""
    return r * n * n + c * n + v + 1


def alo(lits):
    return [list(lits)]


# ---------- AMO encodings: all share the signature (lits, next_var) -> (clauses, next_var) ----------
def pairwise_amo(lits, next_var=None):
    clauses = []
    for i in range(len(lits)):
        for j in range(i + 1, len(lits)):
            clauses.append([-lits[i], -lits[j]])
    return clauses, next_var


def sequential_amo(lits, next_var):
    n = len(lits)
    if n <= 1:
        return [], next_var

    s = list(range(next_var, next_var + n - 1))
    clauses = [[-lits[0], s[0]]]
    for i in range(1, n - 1):
        clauses.append([-lits[i], s[i]])
        clauses.append([-s[i - 1], s[i]])
        clauses.append([-lits[i], -s[i - 1]])
    clauses.append([-lits[n - 1], -s[n - 2]])

    return clauses, next_var + n - 1


def commander_amo(lits, next_var, group_size=3):
    n = len(lits)
    if n <= 1:
        return [], next_var
    if n <= group_size:
        return pairwise_amo(lits)[0], next_var

    clauses, commanders = [], []
    for i in range(0, n, group_size):
        group = lits[i:i + group_size]
        if len(group) == 1:
            commanders.append(group[0])
            continue
        clauses += pairwise_amo(group)[0]
        c = next_var
        next_var += 1
        clauses += [[-l, c] for l in group]
        commanders.append(c)

    sub, next_var = commander_amo(commanders, next_var, group_size)
    return clauses + sub, next_var


def product_amo(lits, next_var, base=6):
    n = len(lits)
    if n <= 1:
        return [], next_var
    if n <= base:
        return pairwise_amo(lits)[0], next_var

    p = ceil(sqrt(n))
    q = ceil(n / p)

    u = list(range(next_var, next_var + p))
    next_var += p
    v = list(range(next_var, next_var + q))
    next_var += q

    clauses = []
    for idx, x in enumerate(lits):
        i, j = divmod(idx, q)
        clauses.append([-x, u[i]])
        clauses.append([-x, v[j]])

    sub_u, next_var = product_amo(u, next_var, base)
    sub_v, next_var = product_amo(v, next_var, base)
    return clauses + sub_u + sub_v, next_var


def binary_amo(lits, next_var):
    n = len(lits)
    if n <= 1:
        return [], next_var

    m = ceil(log2(n))
    bits = list(range(next_var, next_var + m))

    clauses = []
    for i, x in enumerate(lits):
        for k in range(m):
            clauses.append([-x, bits[k] if (i >> k) & 1 else -bits[k]])

    return clauses, next_var + m


# ---------- dispatch ----------
AMO_ENCODINGS = {
    "pairwise": pairwise_amo,
    "sequential": sequential_amo,
    "commander": commander_amo,
    "product": product_amo,
    "binary": binary_amo,
}


def exactly_one(lits, encoding, next_var):
    """ALO + selected AMO encoding."""
    key = encoding.lower()
    if key not in AMO_ENCODINGS:
        raise ValueError(f"encoding must be one of {list(AMO_ENCODINGS)}")

    amo_clauses, next_var = AMO_ENCODINGS[key](lits, next_var)
    return alo(lits) + amo_clauses, next_var


# ---------- Sudoku ----------
def sudoku_clauses(grid, encoding="pairwise"):
    """
    Convert a Sudoku grid into CNF clauses (0 = empty cell).
    encoding: "pairwise", "sequential", "commander", "product", or "binary".

    Returns:
        clauses: CNF clauses
        max_var: largest variable ID used
    """
    n = len(grid)
    box_size = isqrt(n)

    if box_size * box_size != n:
        raise ValueError("Sudoku size must be a perfect square")
    if any(len(row) != n for row in grid):
        raise ValueError("grid must be an n x n matrix")
    if any(value < 0 or value > n for row in grid for value in row):
        raise ValueError(f"Cell values must be between 0 and {n}")
    if encoding.lower() not in AMO_ENCODINGS:
        raise ValueError(f"encoding must be one of {list(AMO_ENCODINGS)}")

    clauses = []
    next_var = n ** 3 + 1  # primary variables occupy 1..n^3

    def add_exactly_one(lits):
        nonlocal next_var
        new_clauses, next_var = exactly_one(lits, encoding, next_var)
        clauses.extend(new_clauses)

    # 1. Cell: exactly one value
    for r in range(n):
        for c in range(n):
            add_exactly_one([var_id(r, c, v, n) for v in range(n)])

    # 2. Row/value: exactly once
    for r in range(n):
        for v in range(n):
            add_exactly_one([var_id(r, c, v, n) for c in range(n)])

    # 3. Column/value: exactly once
    for c in range(n):
        for v in range(n):
            add_exactly_one([var_id(r, c, v, n) for r in range(n)])

    # 4. Box/value: exactly once
    for box_r in range(0, n, box_size):
        for box_c in range(0, n, box_size):
            for v in range(n):
                add_exactly_one([
                    var_id(r, c, v, n)
                    for r in range(box_r, box_r + box_size)
                    for c in range(box_c, box_c + box_size)
                ])

    # 5. Givens
    for r in range(n):
        for c in range(n):
            if grid[r][c] != 0:
                clauses.append([var_id(r, c, grid[r][c] - 1, n)])

    return clauses, next_var - 1


def decode(model, n):
    """Turn a SAT model (list of signed ints) into an n x n grid."""
    true_vars = {l for l in model if l > 0}
    grid = [[0] * n for _ in range(n)]
    for r in range(n):
        for c in range(n):
            for v in range(n):
                if var_id(r, c, v, n) in true_vars:
                    grid[r][c] = v + 1
    return grid

# 8. PySAT Solver

Recommended workflow:

```text
Sudoku instance
      ↓
CNF generator
      ↓
PySAT Solver
      ↓
SAT / UNSAT
      ↓
decode model
      ↓
independent Sudoku verification
```

Start with one baseline solver, such as Glucose, before comparing multiple SAT backends.


In [6]:
# If necessary:
# %pip install python-sat

def solve_cnf(clauses, solver_cls=Glucose3):
    solver = solver_cls()
    try:
        for clause in clauses:
            solver.add_clause(clause)
        sat = solver.solve()
        model = solver.get_model() if sat else None
        return sat, model
    finally:
        solver.delete()


# 9. Decoding and Independent Verification

The decoder must inspect only the primary Sudoku variables. Commander/Product/Sequential encodings may introduce auxiliary variables.

Every returned model must be checked independently for:

1. every cell has exactly one value;
2. every row contains all values exactly once;
3. every column contains all values exactly once;
4. every box contains all values exactly once;
5. every original clue is preserved.

This prevents a bug in the encoding from being mistaken for a successful solver result.


In [7]:
def is_valid_sudoku(grid, original=None):
    n = len(grid)
    k = int(n ** 0.5)
    expected = set(range(1, n + 1))

    if any(set(row) != expected for row in grid):
        return False

    for c in range(n):
        if {grid[r][c] for r in range(n)} != expected:
            return False

    for br in range(0, n, k):
        for bc in range(0, n, k):
            values = {
                grid[r][c]
                for r in range(br, br+k)
                for c in range(bc, bc+k)
            }
            if values != expected:
                return False

    if original is not None:
        for r in range(n):
            for c in range(n):
                if original[r][c] != 0 and original[r][c] != grid[r][c]:
                    return False
    return True

# TODO: implement decode_sudoku(model, n)

def decode_sudoku(model, n):
    """
    Decode a SAT model into an n x n Sudoku grid.

    Args:
        model: List of signed literals returned by the SAT solver.
        n: Sudoku size, e.g. 9 for a 9x9 Sudoku.

    Returns:
        A solved Sudoku grid, or None if model is None.
    """
    if model is None:
        return None

    # Keep only positive literals: variables assigned True.
    true_vars = {lit for lit in model if lit > 0}

    grid = [[0 for _ in range(n)] for _ in range(n)]

    for r in range(n):
        for c in range(n):
            for v in range(n):
                variable = var_id(r, c, v, n)

                if variable in true_vars:
                    grid[r][c] = v + 1
                    break

            # No value assigned to this cell.
            if grid[r][c] == 0:
                raise ValueError(
                    f"No value found for cell ({r}, {c})"
                )

    return grid
    



# 10. Unit Tests

Before benchmarking:

- test a solved Sudoku;
- test a known valid Sudoku with known solution;
- test an inconsistent/UNSAT Sudoku;
- test order-4 Sudoku;
- test each encoding separately.

The same test suite must pass for every encoding.


# 11. Benchmark Dataset

Use a fixed benchmark for every method.

Recommended categories:

| Category | Purpose |
|---|---|
| Easy | correctness/sanity |
| Medium | normal performance |
| Hard | stress test |
| Very hard | distinguish inference/encoding behavior |
| Generalized | scalability |

For every instance store:

- instance ID;
- order;
- clue count;
- difficulty;
- puzzle grid;
- known solution if available.

The paper *Sudoku as a SAT Problem* is especially useful as theoretical motivation for hard Sudoku experiments and for discussing minimal vs extended encodings.


In [8]:
from dataclasses import dataclass

@dataclass
class SudokuInstance:
    instance_id: str
    grid: list
    order: int
    clues: int
    difficulty: str = "unknown"
    known_solution: list | None = None


# 12. Benchmark Metrics

Record for every run:

### Formula
- primary variables
- auxiliary variables
- total variables
- clauses
- binary clauses
- non-binary clauses

### Runtime
- encoding/build time
- solver time
- total time

### Outcome
- SAT / UNSAT
- timeout
- error
- independently verified solution

### Solver statistics
If available:
- decisions
- conflicts
- propagations
- restarts

Use the same timeout, hardware and dataset for all comparisons.


In [9]:
import time

def benchmark_solver(clauses, solver_cls=Glucose3):
    start = time.perf_counter()
    solver = solver_cls()

    try:
        for clause in clauses:
            solver.add_clause(clause)

        build_time = time.perf_counter() - start

        t0 = time.perf_counter()
        sat = solver.solve()
        solve_time = time.perf_counter() - t0
        total_time = time.perf_counter() - start

        return {
            "status": "SAT" if sat else "UNSAT",
            "build_time": build_time,
            "solve_time": solve_time,
            "total_time": total_time,
            "model": solver.get_model() if sat else None,
        }
    finally:
        solver.delete()


In [10]:

import time
import pandas as pd
from pysat.solvers import Glucose3
from pathlib import Path

OUTPUT_DIR = Path("Data")


def count_clues(grid):
    """Count the non-empty cells in a Sudoku puzzle."""
    return sum(
        1
        for row in grid
        for value in row
        if value != 0
    )


def puzzle_key(grid):
    """Create a hashable key to identify duplicate puzzles."""
    return tuple(tuple(row) for row in grid)


def collect_benchmark(
    n_puzzles,
    difficulty,
    size,
    encodings=("pairwise", "sequential", "Commander","Product", "Binary"),
    repeats=1,
    csv_path="sudoku_benchmark.csv"
):
    """
    Generate unique Sudoku puzzles and benchmark each encoding
    on the exact same set of puzzles.

    Returns:
        results_df: one row per puzzle, encoding, and repetition
        puzzles: the fixed collection of generated puzzles
    """
    if n_puzzles < 1 or repeats < 1:
        raise ValueError("n_puzzles and repeats must be positive")

    puzzles = []
    seen = set()

    # Generate a fixed collection of unique puzzles.
    max_attempts = n_puzzles * 20
    attempts = 0

    while len(puzzles) < n_puzzles and attempts < max_attempts:
        attempts += 1
        grid = generate_puzzle(
            difficulty,
            size=size
        )

        key = puzzle_key(grid)

        if key in seen:
            continue

        seen.add(key)
        puzzles.append({
            "instance_id": f"puzzle_{len(puzzles) + 1:03d}",
            "grid": [row[:] for row in grid],
            "order": size,
            "clues": count_clues(grid),
            "difficulty": difficulty
        })

    if len(puzzles) < n_puzzles:
        print(
            f"Warning: generated only {len(puzzles)} "
            f"unique puzzles after {attempts} attempts."
        )

    records = []

    # Use the same puzzles for every encoding.
    for puzzle in puzzles:
        grid = puzzle["grid"]

        for encoding in encodings:
            # Measure Sudoku-to-CNF encoding separately.
            start_encoding = time.perf_counter()

            clauses, max_var = sudoku_clauses(
                grid,
                encoding=encoding
            )

            encoding_time = (
                time.perf_counter() - start_encoding
            )

            # Repeat solver runs on the same CNF.
            for run in range(1, repeats + 1):
                result = benchmark_solver(
                    clauses,
                    solver_cls=Glucose3
                )

                records.append({
                    "instance_id": puzzle["instance_id"],
                    "order": puzzle["order"],
                    "clues": puzzle["clues"],
                    "difficulty": puzzle["difficulty"],
                    "encoding": encoding,
                    "run": run,
                    "num_clauses": len(clauses),
                    "max_var": max_var,
                    "status": result["status"],
                    "encoding_time": encoding_time,
                    "build_time": result["build_time"],
                    "solve_time": result["solve_time"],
                    "total_time": result["total_time"]
                })

    results_df = pd.DataFrame(records)

    csv_path = OUTPUT_DIR / csv_path
    results_df.to_csv(csv_path, index=False)
    
    print(f"Generated puzzles: {len(puzzles)}")
    print(f"Benchmark records: {len(results_df)}")
    print(f"Saved results to: {csv_path}")

    return results_df, puzzles
import random

def collect_benchmark36x36(
    n_puzzles,
    difficulty,
    size,
    encodings=("pairwise", "sequential", "Commander","Product", "Binary"),
    repeats=1,
    csv_path="sudoku_benchmark.csv"
):
    """
    Generate unique Sudoku puzzles and benchmark each encoding
    on the exact same set of puzzles.

    Returns:
        results_df: one row per puzzle, encoding, and repetition
        puzzles: the fixed collection of generated puzzles
    """
    if n_puzzles < 1 or repeats < 1:
        raise ValueError("n_puzzles and repeats must be positive")

    puzzles = []
    seen = set()

    # Generate a fixed collection of unique puzzles.
    max_attempts = n_puzzles * 20
    attempts = 0

    while len(puzzles) < n_puzzles and attempts < max_attempts:
        attempts += 1
        num = random.randint(301, 500)
        grid = generate_puzzle36x36(givens=num 
        )

        key = puzzle_key(grid)

        if key in seen:
            continue

        seen.add(key)
        puzzles.append({
            "instance_id": f"puzzle_{len(puzzles) + 1:03d}",
            "grid": [row[:] for row in grid],
            "order": size,
            "clues": count_clues(grid),
            "difficulty": difficulty
        })

    if len(puzzles) < n_puzzles:
        print(
            f"Warning: generated only {len(puzzles)} "
            f"unique puzzles after {attempts} attempts."
        )

    records = []

    # Use the same puzzles for every encoding.
    for puzzle in puzzles:
        grid = puzzle["grid"]

        for encoding in encodings:
            # Measure Sudoku-to-CNF encoding separately.
            start_encoding = time.perf_counter()

            clauses, max_var = sudoku_clauses(
                grid,
                encoding=encoding
            )

            encoding_time = (
                time.perf_counter() - start_encoding
            )

            # Repeat solver runs on the same CNF.
            for run in range(1, repeats + 1):
                result = benchmark_solver(
                    clauses,
                    solver_cls=Glucose3
                )

                records.append({
                    "instance_id": puzzle["instance_id"],
                    "order": puzzle["order"],
                    "clues": puzzle["clues"],
                    "difficulty": puzzle["difficulty"],
                    "encoding": encoding,
                    "run": run,
                    "num_clauses": len(clauses),
                    "max_var": max_var,
                    "status": result["status"],
                    "encoding_time": encoding_time,
                    "build_time": result["build_time"],
                    "solve_time": result["solve_time"],
                    "total_time": result["total_time"]
                })

    results_df = pd.DataFrame(records)

    csv_path = OUTPUT_DIR / csv_path
    results_df.to_csv(csv_path, index=False)
    
    print(f"Generated puzzles: {len(puzzles)}")
    print(f"Benchmark records: {len(results_df)}")
    print(f"Saved results to: {csv_path}")

    return results_df, puzzles


In [11]:

"""
results_df36x36, puzzles36x36 = collect_benchmark36x36(
    n_puzzles=60,
    difficulty=2,
    size=9,
    encodings=("pairwise", "sequential", "Commander","Product", "Binary"),
    repeats=1,
    csv_path="sudoku_36x36_benchmark.csv"
)
"""


'\nresults_df36x36, puzzles36x36 = collect_benchmark36x36(\n    n_puzzles=60,\n    difficulty=2,\n    size=9,\n    encodings=("pairwise", "sequential", "Commander","Product", "Binary"),\n    repeats=1,\n    csv_path="sudoku_36x36_benchmark.csv"\n)\n'

In [12]:


def plot_benchmark(results_df, save_prefix="sudoku_benchmark"):
    """
    Plot and save benchmark comparisons for each encoding.
    """
    if results_df.empty:
        raise ValueError("Benchmark DataFrame is empty")

    sns.set_theme(style="whitegrid")

    # 1. Solver runtime distribution.
    plt.figure(figsize=(9, 5))
    sns.boxplot(
        data=results_df,
        x="encoding",
        y="solve_time"
    )
    plt.title("SAT Solver Runtime by Encoding")
    plt.xlabel("Encoding")
    plt.ylabel("Solve time (seconds)")
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / f"{save_prefix}_solve_time.png", dpi=150,)
    plt.show()

    # 2. CNF encoding time.
    # Encoding time is measured once per puzzle and encoding,
    # so remove repeated solver-run records before plotting.
    encoding_data = results_df.drop_duplicates(
        subset=["instance_id", "encoding"]
    )

    plt.figure(figsize=(9, 5))
    sns.boxplot(
        data=encoding_data,
        x="encoding",
        y="encoding_time"
    )
    plt.title("CNF Encoding Time by Method")
    plt.xlabel("Encoding")
    plt.ylabel("Encoding time (seconds)")
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / f"{save_prefix}_encoding_time.png", dpi=150)
    plt.show()

    # 3. Number of clauses.
    plt.figure(figsize=(9, 5))
    sns.barplot(
        data=encoding_data,
        x="encoding",
        y="num_clauses",
        errorbar=None
    )
    plt.title("Number of CNF Clauses by Encoding")
    plt.xlabel("Encoding")
    plt.ylabel("Number of clauses")
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / f"{save_prefix}_clauses.png", dpi=150)
    plt.show()

    # 4. Mean solve time for each encoding.
    summary = (
        results_df
        .groupby("encoding", as_index=False)
        .agg(
            mean_solve_time=("solve_time", "mean"),
            median_solve_time=("solve_time", "median"),
            mean_build_time=("build_time", "mean"),
            mean_total_time=("total_time", "mean"),
            mean_clauses=("num_clauses", "mean"),
            mean_max_var=("max_var", "mean")
        )
    )
    

    return summary


In [13]:
n = generate_puzzle36x36()
print(n)


[[29, 0, 0, 0, 12, 0, 36, 0, 0, 0, 0, 16, 0, 0, 0, 26, 30, 20, 9, 0, 25, 0, 1, 0, 0, 13, 0, 5, 11, 0, 6, 0, 32, 0, 0, 0], [28, 6, 0, 0, 0, 0, 0, 0, 0, 0, 11, 0, 23, 0, 0, 9, 0, 0, 0, 0, 0, 7, 20, 0, 18, 0, 17, 0, 0, 0, 8, 0, 0, 0, 0, 0], [0, 14, 0, 0, 17, 0, 22, 0, 0, 0, 26, 0, 0, 0, 27, 6, 0, 0, 0, 29, 4, 0, 0, 12, 0, 0, 0, 0, 9, 1, 0, 10, 5, 13, 0, 19], [13, 0, 5, 0, 10, 19, 0, 0, 0, 0, 9, 0, 0, 0, 0, 8, 0, 0, 0, 28, 27, 0, 0, 24, 0, 22, 0, 0, 26, 0, 0, 0, 0, 0, 0, 18], [0, 0, 0, 0, 31, 25, 0, 3, 4, 0, 8, 0, 0, 0, 18, 0, 0, 2, 0, 0, 19, 0, 0, 0, 0, 0, 0, 32, 0, 0, 0, 0, 7, 0, 0, 0], [0, 26, 7, 0, 0, 0, 28, 0, 0, 0, 0, 0, 0, 0, 19, 0, 0, 21, 0, 36, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 9, 0, 0, 0, 0, 25], [35, 21, 10, 0, 23, 9, 34, 0, 0, 29, 1, 0, 0, 12, 0, 3, 36, 18, 33, 0, 0, 24, 0, 0, 6, 0, 28, 0, 0, 27, 0, 22, 17, 0, 0, 26], [32, 0, 0, 27, 0, 6, 0, 19, 0, 13, 0, 0, 0, 0, 9, 0, 23, 0, 0, 0, 0, 0, 0, 22, 0, 0, 0, 0, 0, 0, 1, 29, 0, 34, 0, 0], [0, 3, 12, 18, 0, 14, 0, 15, 0, 22, 0, 17, 32, 0

In [14]:
print(n)

[[29, 0, 0, 0, 12, 0, 36, 0, 0, 0, 0, 16, 0, 0, 0, 26, 30, 20, 9, 0, 25, 0, 1, 0, 0, 13, 0, 5, 11, 0, 6, 0, 32, 0, 0, 0], [28, 6, 0, 0, 0, 0, 0, 0, 0, 0, 11, 0, 23, 0, 0, 9, 0, 0, 0, 0, 0, 7, 20, 0, 18, 0, 17, 0, 0, 0, 8, 0, 0, 0, 0, 0], [0, 14, 0, 0, 17, 0, 22, 0, 0, 0, 26, 0, 0, 0, 27, 6, 0, 0, 0, 29, 4, 0, 0, 12, 0, 0, 0, 0, 9, 1, 0, 10, 5, 13, 0, 19], [13, 0, 5, 0, 10, 19, 0, 0, 0, 0, 9, 0, 0, 0, 0, 8, 0, 0, 0, 28, 27, 0, 0, 24, 0, 22, 0, 0, 26, 0, 0, 0, 0, 0, 0, 18], [0, 0, 0, 0, 31, 25, 0, 3, 4, 0, 8, 0, 0, 0, 18, 0, 0, 2, 0, 0, 19, 0, 0, 0, 0, 0, 0, 32, 0, 0, 0, 0, 7, 0, 0, 0], [0, 26, 7, 0, 0, 0, 28, 0, 0, 0, 0, 0, 0, 0, 19, 0, 0, 21, 0, 36, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 9, 0, 0, 0, 0, 25], [35, 21, 10, 0, 23, 9, 34, 0, 0, 29, 1, 0, 0, 12, 0, 3, 36, 18, 33, 0, 0, 24, 0, 0, 6, 0, 28, 0, 0, 27, 0, 22, 17, 0, 0, 26], [32, 0, 0, 27, 0, 6, 0, 19, 0, 13, 0, 0, 0, 0, 9, 0, 23, 0, 0, 0, 0, 0, 0, 22, 0, 0, 0, 0, 0, 0, 1, 29, 0, 34, 0, 0], [0, 3, 12, 18, 0, 14, 0, 15, 0, 22, 0, 17, 32, 0

In [15]:
def solve_cplex_cp(grid):

    start = time.perf_counter()

    n = len(grid)
    box = int(n ** 0.5)

    if box * box != n:
        raise ValueError("Sudoku size must be a perfect square.")

    model = CpoModel()

    # x[r][c] là giá trị của ô (r, c), từ 1 đến n
    x = [
        [
            model.integer_var(min=1, max=n, name=f"x_{r}_{c}")
            for c in range(n)
        ]
        for r in range(n)
    ]

    # Mỗi hàng và cột chứa các giá trị khác nhau
    for r in range(n):
        model.add(model.all_diff(x[r]))

    for c in range(n):
        model.add(model.all_diff([x[r][c] for r in range(n)]))

    # Mỗi box chứa các giá trị khác nhau
    for br in range(0, n, box):
        for bc in range(0, n, box):
            cells = [
                x[r][c]
                for r in range(br, br + box)
                for c in range(bc, bc + box)
            ]
            model.add(model.all_diff(cells))

    # Giữ nguyên các ô đã cho
    for r in range(n):
        for c in range(n):
            if grid[r][c] != 0:
                model.add(x[r][c] == grid[r][c])

    build_time = time.perf_counter() - start

    solve_start = time.perf_counter()
    solution = model.solve(TimeLimit=60)
    solve_time = time.perf_counter() - solve_start

    status = "UNSAT"
    solved_grid = None

    if solution is not None:
        solved_grid = [
            [solution.get_value(x[r][c]) for c in range(n)]
            for r in range(n)
        ]
        status = "SAT"

    return {
        "method": "CPLEX CP",
        "status": status,
        "build_time": build_time,
        "solve_time": solve_time,
        "total_time": build_time + solve_time,
        "solution": solved_grid,
    }


def solve_cplex_mip(grid):

    start = time.perf_counter()

    n = len(grid)
    box = int(n ** 0.5)

    if box * box != n:
        raise ValueError("Sudoku size must be a perfect square.")

    model = Model(name="Sudoku_MIP")

    # x[r,c,v] = 1 nếu ô (r,c) nhận giá trị v+1
    x = {
        (r, c, v): model.binary_var(name=f"x_{r}_{c}_{v+1}")
        for r in range(n)
        for c in range(n)
        for v in range(n)
    }

    # Mỗi ô có đúng một giá trị
    for r in range(n):
        for c in range(n):
            model.add_constraint(
                model.sum(x[r, c, v] for v in range(n)) == 1
            )

    # Mỗi giá trị xuất hiện đúng một lần trong mỗi hàng
    for r in range(n):
        for v in range(n):
            model.add_constraint(
                model.sum(x[r, c, v] for c in range(n)) == 1
            )

    # Mỗi giá trị xuất hiện đúng một lần trong mỗi cột
    for c in range(n):
        for v in range(n):
            model.add_constraint(
                model.sum(x[r, c, v] for r in range(n)) == 1
            )

    # Mỗi giá trị xuất hiện đúng một lần trong mỗi box
    for br in range(0, n, box):
        for bc in range(0, n, box):
            for v in range(n):
                model.add_constraint(
                    model.sum(
                        x[r, c, v]
                        for r in range(br, br + box)
                        for c in range(bc, bc + box)
                    ) == 1
                )

    # Các ô đã cho
    for r in range(n):
        for c in range(n):
            if grid[r][c] != 0:
                v = grid[r][c] - 1
                model.add_constraint(x[r, c, v] == 1)

    # Sudoku chỉ cần tìm nghiệm khả thi, không cần hàm mục tiêu
    model.minimize(0)

    build_time = time.perf_counter() - start

    solve_start = time.perf_counter()
    solution = model.solve(log_output=False)
    solve_time = time.perf_counter() - solve_start

    solved_grid = None
    status = "UNSAT"

    if solution is not None:
        solved_grid = [
            [
                next(
                    v + 1
                    for v in range(n)
                    if solution.get_value(x[r, c, v]) > 0.5
                )
                for c in range(n)
            ]
            for r in range(n)
        ]
        status = "SAT"

    return {
        "method": "CPLEX MIP",
        "status": status,
        "build_time": build_time,
        "solve_time": solve_time,
        "total_time": build_time + solve_time,
        "solution": solved_grid,
    }

def solve_gurobi(grid):



    start = time.perf_counter()

    n = len(grid)
    box = int(n ** 0.5)

    if box * box != n:
        raise ValueError("Sudoku size must be a perfect square.")

    model = gp.Model("Sudoku_Gurobi")
    model.Params.OutputFlag = 0
    model.Params.TimeLimit = 60

    # x[r,c,v] = 1 nếu ô (r,c) nhận giá trị v+1
    x = model.addVars(
        n, n, n,
        vtype=GRB.BINARY,
        name="x"
    )

    # Mỗi ô có đúng một giá trị
    for r in range(n):
        for c in range(n):
            model.addConstr(
                gp.quicksum(x[r, c, v] for v in range(n)) == 1
            )

    # Hàng
    for r in range(n):
        for v in range(n):
            model.addConstr(
                gp.quicksum(x[r, c, v] for c in range(n)) == 1
            )

    # Cột
    for c in range(n):
        for v in range(n):
            model.addConstr(
                gp.quicksum(x[r, c, v] for r in range(n)) == 1
            )

    # Box
    for br in range(0, n, box):
        for bc in range(0, n, box):
            for v in range(n):
                model.addConstr(
                    gp.quicksum(
                        x[r, c, v]
                        for r in range(br, br + box)
                        for c in range(bc, bc + box)
                    ) == 1
                )

    # Ô đã cho
    for r in range(n):
        for c in range(n):
            if grid[r][c] != 0:
                model.addConstr(x[r, c, grid[r][c] - 1] == 1)

    model.setObjective(0, GRB.MINIMIZE)

    build_time = time.perf_counter() - start

    solve_start = time.perf_counter()
    model.optimize()
    solve_time = time.perf_counter() - solve_start

    solved_grid = None
    if model.SolCount > 0:
        solved_grid = [
            [
                next(
                    v + 1 for v in range(n)
                    if x[r, c, v].X > 0.5
                )
                for c in range(n)
            ]
            for r in range(n)
        ]

    if model.Status == GRB.INFEASIBLE:
        status = "UNSAT"
    elif model.SolCount > 0:
        status = "SAT"
    elif model.Status == GRB.TIME_LIMIT:
        status = "UNKNOWN"
    else:
        status = "UNKNOWN"

    result = {
        "method": "Gurobi",
        "status": status,
        "build_time": build_time,
        "solve_time": solve_time,
        "total_time": build_time + solve_time,
        "solution": solved_grid,
    }

    model.dispose()
    return result

def solve_cp_sat(grid):


    start = time.perf_counter()

    n = len(grid)
    box = int(n ** 0.5)

    if box * box != n:
        raise ValueError("Sudoku size must be a perfect square.")

    model = cp_model.CpModel()

    x = [
        [
            model.new_int_var(1, n, f"x_{r}_{c}")
            for c in range(n)
        ]
        for r in range(n)
    ]

    # Hàng và cột
    for r in range(n):
        model.add_all_different(x[r])

    for c in range(n):
        model.add_all_different([x[r][c] for r in range(n)])

    # Box
    for br in range(0, n, box):
        for bc in range(0, n, box):
            cells = [
                x[r][c]
                for r in range(br, br + box)
                for c in range(bc, bc + box)
            ]
            model.add_all_different(cells)

    # Các ô đã cho
    for r in range(n):
        for c in range(n):
            if grid[r][c] != 0:
                model.add(x[r][c] == grid[r][c])

    build_time = time.perf_counter() - start

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 60

    solve_start = time.perf_counter()
    status_code = solver.solve(model)
    solve_time = time.perf_counter() - solve_start

    if status_code in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        status = "SAT"
        solved_grid = [
            [solver.value(x[r][c]) for c in range(n)]
            for r in range(n)
        ]
    elif status_code == cp_model.INFEASIBLE:
        status = "UNSAT"
        solved_grid = None
    else:
        # UNKNOWN: ví dụ hết thời gian mà chưa chứng minh được
        # có nghiệm hoặc vô nghiệm.
        status = "UNKNOWN"
        solved_grid = None

    return {
        "method": "CP-SAT",
        "status": status,
        "build_time": build_time,
        "solve_time": solve_time,
        "total_time": build_time + solve_time,
        "solution": solved_grid,
    }

In [16]:
test_grid = generate_puzzle36x36(givens= 500)
#test_grid = generate_puzzle(1, size= 25)

solvers = [
    #solve_cplex_cp,
    #solve_cplex_mip,
    #solve_gurobi,
    solve_cp_sat
]

benchmark_results = []

for solve_function in solvers:
    try:
        result = solve_function(test_grid)
        benchmark_results.append(result)

        print(
            result["method"],
            "| Status:", result["status"],
            "| Build:", round(result["build_time"], 6),
            "| Solve:", round(result["solve_time"], 6),
            "| Total:", round(result["total_time"], 6),
        )

        if result["solution"] is not None:
            for row in result["solution"]:
                print(row)

    except Exception as error:
        print(
            solve_function.__name__,
            "failed:",
            repr(error)
        )

CP-SAT | Status: SAT | Build: 0.051163 | Solve: 8.855122 | Total: 8.906284
[23, 6, 15, 33, 21, 3, 22, 18, 9, 19, 7, 31, 26, 8, 27, 28, 5, 1, 34, 10, 11, 16, 30, 17, 35, 12, 2, 32, 24, 4, 20, 25, 13, 29, 14, 36]
[8, 18, 5, 36, 1, 24, 17, 20, 32, 30, 10, 33, 16, 3, 25, 29, 13, 15, 31, 35, 12, 26, 27, 2, 23, 6, 9, 34, 11, 14, 19, 4, 28, 22, 21, 7]
[9, 14, 17, 20, 31, 10, 1, 5, 16, 35, 34, 27, 11, 7, 2, 36, 33, 6, 13, 25, 24, 3, 4, 29, 22, 21, 18, 26, 28, 19, 8, 30, 23, 12, 15, 32]
[12, 2, 4, 27, 11, 26, 25, 14, 36, 29, 8, 3, 24, 34, 21, 23, 35, 10, 7, 22, 28, 19, 32, 18, 15, 30, 20, 31, 13, 33, 17, 1, 9, 6, 5, 16]
[25, 16, 30, 35, 29, 7, 23, 13, 28, 2, 4, 11, 32, 22, 19, 12, 20, 18, 15, 33, 21, 14, 6, 9, 8, 27, 1, 5, 17, 36, 26, 34, 3, 10, 24, 31]
[19, 32, 13, 22, 34, 28, 21, 15, 6, 12, 26, 24, 17, 4, 30, 14, 31, 9, 20, 8, 23, 5, 36, 1, 10, 7, 29, 25, 3, 16, 11, 2, 35, 18, 33, 27]
[34, 3, 26, 9, 7, 27, 16, 25, 21, 18, 13, 29, 33, 14, 11, 15, 6, 23, 22, 24, 17, 32, 19, 28, 31, 36, 10, 4, 2

In [31]:
import time
import pandas as pd
from pathlib import Path
from pysat.solvers import Glucose3

OUTPUT_DIR = Path("Data")


def collect_benchmark_all_methods(
    n_puzzles=60,
    difficulty=1,
    size=9,
    repeats=1,
    sat_encodings=(
        "pairwise",
        "sequential",
        "commander",
        "product",
        "binary",
    ),
    csv_path="sudoku_benchmark_all_methods.csv",
):
    """
    Benchmark nine Sudoku-solving methods on the same puzzles.

    SAT encodings:
        Pairwise, Sequential, Commander, Product, Binary

    Standalone solvers:
        CPLEX CP, CPLEX MIP, Gurobi, CP-SAT

    Returns:
        results_df: one row per puzzle, method, and repetition
        puzzles: fixed collection of unique Sudoku puzzles
    """

    if n_puzzles < 1 or repeats < 1:
        raise ValueError("n_puzzles and repeats must be positive")

    sat_encodings = tuple(sat_encodings)

    supported_encodings = {
        "pairwise",
        "sequential",
        "commander",
        "product",
        "binary",
    }

    invalid_encodings = set(sat_encodings) - supported_encodings

    if invalid_encodings:
        raise ValueError(
            f"Unsupported SAT encodings: {invalid_encodings}"
        )

    methods = {
        "CPLEX CP": solve_cplex_cp,
        "CPLEX MIP": solve_cplex_mip,
        "Gurobi": solve_gurobi,
        "CP-SAT": solve_cp_sat,
    }

    # --------------------------------------------------
    # 1. Generate unique puzzles
    # --------------------------------------------------
    puzzles = []
    seen = set()

    max_attempts = n_puzzles * 20
    attempts = 0

    while len(puzzles) < n_puzzles and attempts < max_attempts:
        attempts += 1

        grid = generate_puzzle(
            difficulty=difficulty,
            size=size,
        )

        key = puzzle_key(grid)

        if key in seen:
            continue

        seen.add(key)

        puzzles.append({
            "instance_id": f"puzzle_{len(puzzles) + 1:03d}",
            "grid": [row[:] for row in grid],
            "order": len(grid),
            "clues": count_clues(grid),
            "difficulty": difficulty,
        })

    if len(puzzles) < n_puzzles:
        print(
            f"Warning: generated only {len(puzzles)} unique puzzles "
            f"after {attempts} attempts."
        )

    records = []

    # --------------------------------------------------
    # 2. Benchmark all methods on the same puzzles
    # --------------------------------------------------
    for puzzle in puzzles:
        grid = puzzle["grid"]

        common_info = {
            "instance_id": puzzle["instance_id"],
            "order": puzzle["order"],
            "clues": puzzle["clues"],
            "difficulty": puzzle["difficulty"],
        }

        # ----------------------------------------------
        # 2A. Five SAT encodings
        # ----------------------------------------------
        for encoding in sat_encodings:

            for run in range(1, repeats + 1):
                encoding_time = None
                build_time = None
                solve_time = None
                total_time = None
                status = "ERROR"
                error_message = None

                try:
                    # Build the CNF for this encoding
                    encoding_start = time.perf_counter()

                    clauses, max_var = sudoku_clauses(
                        grid,
                        encoding=encoding,
                    )

                    encoding_time = (
                        time.perf_counter() - encoding_start
                    )

                    # Solve the generated CNF
                    result = benchmark_solver(
                        clauses,
                        solver_cls=Glucose3,
                    )

                    status = result["status"]
                    build_time = (
                        encoding_time + result["build_time"]
                    )
                    solve_time = result["solve_time"]
                    total_time = build_time + solve_time

                except Exception as error:
                    error_message = repr(error)

                records.append({
                    **common_info,
                    "method": f"SAT-{encoding}",
                    "run": run,
                    "status": status,
                    "encoding_time": encoding_time,
                    "build_time": build_time,
                    "solve_time": solve_time,
                    "total_time": total_time,
                    "error": error_message,
                })

        # ----------------------------------------------
        # 2B. Four standalone solvers
        # ----------------------------------------------
        for method_name, solve_function in methods.items():

            for run in range(1, repeats + 1):
                try:
                    result = solve_function(grid)

                    build_time = result["build_time"]
                    solve_time = result["solve_time"]

                    records.append({
                        **common_info,
                        "method": method_name,
                        "run": run,
                        "status": result["status"],
                        "encoding_time": None,
                        "build_time": build_time,
                        "solve_time": solve_time,
                        "total_time": build_time + solve_time,
                        "error": None,
                    })

                except Exception as error:
                    records.append({
                        **common_info,
                        "method": method_name,
                        "run": run,
                        "status": "ERROR",
                        "encoding_time": None,
                        "build_time": None,
                        "solve_time": None,
                        "total_time": None,
                        "error": repr(error),
                    })

    # --------------------------------------------------
    # 3. Save combined results
    # --------------------------------------------------
    results_df = pd.DataFrame(records)

    output_path = OUTPUT_DIR / csv_path
    results_df.to_csv(output_path, index=False)

    print(f"Generated puzzles: {len(puzzles)}")
    print(f"Methods requested: {len(sat_encodings) + len(methods)}")
    print(f"Benchmark records: {len(results_df)}")
    print(f"Saved results to: {output_path.resolve()}")

    print("\nResults by method:")
    print(results_df.groupby("method")["status"].value_counts())

    return results_df, puzzles


def collect_benchmark_all_methods36x36(
    n_puzzles=60,
    difficulty=1,
    size=9,
    repeats=1,
    sat_encodings=(""),
    csv_path="sudoku_benchmark_all_methods36x36.csv",
):
    """
    Benchmark nine Sudoku-solving methods on the same puzzles.

    SAT encodings:
        Pairwise, Sequential, Commander, Product, Binary

    Standalone solvers:
        CPLEX CP, CPLEX MIP, Gurobi, CP-SAT

    Returns:
        results_df: one row per puzzle, method, and repetition
        puzzles: fixed collection of unique Sudoku puzzles
    """

    if n_puzzles < 1 or repeats < 1:
        raise ValueError("n_puzzles and repeats must be positive")
    """
    sat_encodings = tuple(sat_encodings)

    supported_encodings = {
        
    }

    invalid_encodings = set(sat_encodings) - supported_encodings

    if invalid_encodings:
        raise ValueError(
            f"Unsupported SAT encodings: {invalid_encodings}"
        )"""

    methods = {

        "CP-SAT": solve_cp_sat,
    }

    # --------------------------------------------------
    # 1. Generate unique puzzles
    # --------------------------------------------------
    puzzles = []
    seen = set()

    max_attempts = n_puzzles * 20
    attempts = 0

    while len(puzzles) < n_puzzles and attempts < max_attempts:
        attempts += 1
        num = random.randint(301, 500)
        grid = generate_puzzle36x36(givens=num 
        )

        key = puzzle_key(grid)

        if key in seen:
            continue

        seen.add(key)

        puzzles.append({
            "instance_id": f"puzzle_{len(puzzles) + 1:03d}",
            "grid": [row[:] for row in grid],
            "order": len(grid),
            "clues": count_clues(grid),
            "difficulty": difficulty,
        })

    if len(puzzles) < n_puzzles:
        print(
            f"Warning: generated only {len(puzzles)} unique puzzles "
            f"after {attempts} attempts."
        )

    records = []

    # --------------------------------------------------
    # 2. Benchmark all methods on the same puzzles
    # --------------------------------------------------
    for puzzle in puzzles:
        grid = puzzle["grid"]

        common_info = {
            "instance_id": puzzle["instance_id"],
            "order": puzzle["order"],
            "clues": puzzle["clues"],
            "difficulty": puzzle["difficulty"],
        }

        # ----------------------------------------------
        # 2A. Five SAT encodings
        # ----------------------------------------------
        """
        for encoding in sat_encodings:

            for run in range(1, repeats + 1):
                encoding_time = None
                build_time = None
                solve_time = None
                total_time = None
                status = "ERROR"
                error_message = None

                try:
                    # Build the CNF for this encoding
                    encoding_start = time.perf_counter()

                    clauses, max_var = sudoku_clauses(
                        grid,
                        encoding=encoding,
                    )

                    encoding_time = (
                        time.perf_counter() - encoding_start
                    )

                    # Solve the generated CNF
                    result = benchmark_solver(
                        clauses,
                        solver_cls=Glucose3,
                    )

                    status = result["status"]
                    build_time = (
                        encoding_time + result["build_time"]
                    )
                    solve_time = result["solve_time"]
                    total_time = build_time + solve_time

                except Exception as error:
                    error_message = repr(error)

                records.append({
                    **common_info,
                    "method": f"SAT-{encoding}",
                    "run": run,
                    "status": status,
                    "encoding_time": encoding_time,
                    "build_time": build_time,
                    "solve_time": solve_time,
                    "total_time": total_time,
                    "error": error_message,
                })"""

        # ----------------------------------------------
        # 2B. Four standalone solvers
        # ----------------------------------------------
        for method_name, solve_function in methods.items():

            for run in range(1, repeats + 1):
                try:
                    result = solve_function(grid)

                    build_time = result["build_time"]
                    solve_time = result["solve_time"]

                    records.append({
                        **common_info,
                        "method": method_name,
                        "run": run,
                        "status": result["status"],
                        "encoding_time": None,
                        "build_time": build_time,
                        "solve_time": solve_time,
                        "total_time": build_time + solve_time,
                        "error": None,
                    })

                except Exception as error:
                    records.append({
                        **common_info,
                        "method": method_name,
                        "run": run,
                        "status": "ERROR",
                        "encoding_time": None,
                        "build_time": None,
                        "solve_time": None,
                        "total_time": None,
                        "error": repr(error),
                    })

    # --------------------------------------------------
    # 3. Save combined results
    # --------------------------------------------------
    results_df = pd.DataFrame(records)

    output_path = OUTPUT_DIR / csv_path
    results_df.to_csv(output_path, index=False)

    print(f"Generated puzzles: {len(puzzles)}")
    print(f"Methods requested: {len(sat_encodings) + len(methods)}")
    print(f"Benchmark records: {len(results_df)}")
    print(f"Saved results to: {output_path.resolve()}")

    print("\nResults by method:")
    print(results_df.groupby("method")["status"].value_counts())

    return results_df, puzzles

In [32]:

results_df, puzzles = collect_benchmark_all_methods36x36(
    n_puzzles=20,
    difficulty=1,
    repeats=1,
    csv_path="sudoku_benchmark_all_methods36x36.csv",
    
)




Generated puzzles: 20
Methods requested: 1
Benchmark records: 20
Saved results to: C:\Users\ACER\PycharmProjects\INT3011E_BTL1\Data\sudoku_benchmark_all_methods36x36.csv

Results by method:
method  status
CP-SAT  SAT       20
Name: count, dtype: int64


# 13. Experiment A — Compare ALO/AMO Encodings

### Independent variable
Encoding:
- Pairwise
- Commander
- Product
- Sequential Counter

### Dependent variables
- variable count
- clause count
- encoding time
- solving time
- total time
- success rate

### Controlled variables
- same Sudoku instances
- same SAT solver
- same machine
- same timeout

This is the main experiment for the assignment.


In [19]:
ENCODINGS = [
    "pairwise",
    "commander",
    "product",
    "sequential",
]

encoding_results = []

# TODO:
# for instance in benchmark:
#     for encoding in ENCODINGS:
#         generate CNF
#         solve
#         verify
#         record metrics
print("hello motherfucker ")

hello motherfucker 


# 14. Experiment B — Compare SAT Solvers

Potential SAT solvers depend on what is actually available in the installed PySAT environment.

Candidates may include:
- Glucose
- CaDiCaL
- Kissat
- CryptoMiniSat
- Minisat / other available backends

For a fair test, fix the encoding and dataset, then change only the solver.

**Do not report a solver as tested unless it actually ran successfully.**


In [20]:
# TODO: inspect the installed PySAT version and available solver backends.
# Then create a dictionary of actually available solver classes.

SOLVERS = {
    "Glucose3": Glucose3,
    # "CaDiCaL": ...,
    # "Kissat": ...,
    # "CryptoMiniSat": ...,
}


# 15. Experiment C — Scalability

If reliable generalized Sudoku instances are available, test:

$$
4 \times 4, \quad 9 \times 9, \quad 16 \times 16
$$

Primary variables grow as:

$$
n^3.
$$

Examples:

$$
4^3 = 64, \quad 9^3 = 729, \quad 16^3 = 4096.
$$

Measure:
- formula size;
- encoding time;
- solving time;
- total time;
- memory/timeout behavior.

Do not assume linear runtime growth. The purpose of this section is to observe it experimentally.


# 16. Experiment D — SAT vs CP-SAT

Implement the same Sudoku model in OR-Tools CP-SAT if available.

Use the same Boolean decision-variable interpretation where practical:

$$
x_{r,c,v} \in \{0, 1\}
$$

with Exactly-One constraints.

Compare:
- SAT/PySAT
- CP-SAT

Use the same instances and timeout.


In [21]:
# Optional:
# %pip install ortools

# TODO:
# from ortools.sat.python import cp_model
#
# def solve_with_cp_sat(grid):
#     ...


# 17. Experiment E — ILP / MIP

A 0–1 MIP formulation can use:

$$
x_{r,c,v} \in \{0, 1\}.
$$

Constraints:

$$
\sum_{v} x_{r,c,v} = 1
$$

for cells,

$$
\sum_{c} x_{r,c,v} = 1
$$

for rows,

$$
\sum_{r} x_{r,c,v} = 1
$$

for columns, and the analogous equation for each box.

If available, compare:
- Gurobi
- CPLEX MIP

The assignment also mentions CPLEX CP; implement it separately if the required environment is available.

If a commercial solver/license is unavailable, document that limitation rather than inventing results.


# 18. Unified Results Table

Use one row per method/instance/run.

Recommended columns:

```text
instance
order
clues
method
solver
encoding
variables
aux_variables
clauses
constraints
build_time
solve_time
total_time
status
valid
```

Save raw results as CSV so the report figures can be regenerated.


In [22]:
import pandas as pd

RESULT_COLUMNS = [
    "instance", "order", "clues",
    "method", "solver", "encoding",
    "variables", "aux_variables", "clauses", "constraints",
    "build_time", "solve_time", "total_time",
    "status", "valid"
]

results = pd.DataFrame(columns=RESULT_COLUMNS)
results.head()


,instance,order,clues,method,solver,encoding,variables,aux_variables,clauses,constraints,build_time,solve_time,total_time,status,valid


# 19. Data Analysis

Answer these questions using measured data:

### Encoding
- Which encoding has the smallest CNF?
- Which has the smallest number of auxiliary variables?
- Which is fastest?
- Does the best encoding change with puzzle size?

### SAT solver
- Which solver is fastest?
- Which is most consistent?
- Which handles hard instances best?

### Cross-paradigm
- SAT vs CP-SAT?
- SAT vs MIP?
- SAT vs CPLEX CP?
- SAT vs Gurobi?
- Does the ranking change for larger instances?

### Scalability
- How do variables/clauses grow?
- When does encoding time become significant?
- When does solving time dominate?


In [23]:
import matplotlib.pyplot as plt

def plot_median_runtime(df, group_col, title):
    summary = (
        df[df["status"] == "SAT"]
        .groupby(group_col)["solve_time"]
        .median()
        .sort_values()
    )
    summary.plot(kind="bar")
    plt.ylabel("Median solving time (s)")
    plt.title(title)
    plt.tight_layout()
    plt.show()

# Example:
# plot_median_runtime(results, "encoding", "Median Runtime by Encoding")


# 20. Recommended Figures for the Report

Include at least:

1. Clause count by encoding.
2. Variable count by encoding.
3. Median solving time by encoding.
4. SAT solver runtime comparison.
5. Solved percentage by method.
6. Runtime vs Sudoku difficulty.
7. Runtime vs Sudoku order.
8. Optional CDF of solving times.

Use median runtime and solved percentage in addition to mean runtime because hard-instance runtimes can contain strong outliers.


# 21. Discussion

The discussion should explain *why* results occurred.

Important theoretical point from the reference paper:

> A redundant SAT clause can be computationally useful because restricted inference such as unit propagation may exploit an explicitly present consequence much more easily than deriving that consequence through resolution.

For example, the paper's minimal and extended Sudoku encodings represent the same Sudoku solutions, but the extended encoding exposes additional constraints explicitly.

Use this idea when interpreting your own encoding experiments.

Do not assume that the encoding with fewer clauses is automatically the best.


# 22. Reproducibility / GitHub

Recommended repository:

```text
sudoku-sat/
├── notebooks/
│   └── sudoku_sat_experiments.ipynb
├── src/
│   ├── sudoku.py
│   ├── encodings.py
│   ├── solvers.py
│   ├── verification.py
│   └── benchmarks.py
├── data/
├── results/
├── figures/
├── requirements.txt
└── README.md
```

The final report should include the GitHub/other repository link as requested in the assignment.

Record:
- Python version
- OS
- CPU/RAM
- PySAT version
- solver versions
- OR-Tools version
- Gurobi/CPLEX versions if used


In [24]:
import sys
import platform

print("Python:", sys.version)
print("OS:", platform.platform())
print("Processor:", platform.processor())

try:
    import pysat
    print("PySAT:", getattr(pysat, "__version__", "unknown"))
except ImportError:
    print("PySAT: not installed")

try:
    import pandas as pd
    print("pandas:", pd.__version__)
except ImportError:
    print("pandas: not installed")


Python: 3.12.5 (tags/v3.12.5:ff3bc82, Aug  6 2024, 20:45:27) [MSC v.1940 64 bit (AMD64)]
OS: Windows-11-10.0.26300-SP0
Processor: Intel64 Family 6 Model 186 Stepping 2, GenuineIntel
PySAT: 3.2.2
pandas: 2.3.3


# 23. Final Report Structure

The notebook can be converted into the final report using this structure:

## 1. Introduction
- Sudoku as a constraint problem
- Motivation for SAT
- Research questions

## 2. Background
- SAT and CNF
- ALO / AMO
- PySAT
- SAT solver concepts

## 3. Sudoku SAT Formulation
- Boolean variables
- Cell/row/column/box constraints
- Clue encoding

## 4. SAT Encodings
- Pairwise
- Commander
- Product
- Sequential Counter
- Complexity and clause/variable counts

## 5. Implementation
- architecture
- PySAT
- decoding
- verification

## 6. Experimental Methodology
- dataset
- hardware/software
- timeout
- metrics
- controlled variables

## 7. Experimental Results
- encoding comparison
- solver comparison
- scalability
- SAT vs CP/CP-SAT/MIP/CPLEX/Gurobi

## 8. Discussion
- explain observed behavior
- trade-offs
- limitations

## 9. Conclusion
- answer research questions
- summarize main findings

## References
- papers
- documentation
- datasets


# 24. Final Checklist

### Implementation
- [ ] Pairwise AMO works
- [ ] Commander works
- [ ] Product works
- [ ] Sequential Counter works
- [ ] Sudoku CNF is correct
- [ ] Model decoding is correct
- [ ] Independent verifier works
- [ ] Unit tests pass

### Experiments
- [ ] Same benchmark for all methods
- [ ] Same timeout
- [ ] Same hardware
- [ ] Encoding time recorded
- [ ] Solver time recorded
- [ ] Formula size recorded
- [ ] Results exported to CSV
- [ ] Figures generated

### Comparisons
- [ ] SAT encodings
- [ ] SAT solvers
- [ ] CP-SAT
- [ ] CP if available
- [ ] ILP/MIP
- [ ] CPLEX MIP if available
- [ ] CPLEX CP if available
- [ ] Gurobi if available

### Submission
- [ ] Report
- [ ] Notebook
- [ ] Source code
- [ ] Dataset/source
- [ ] GitHub link
- [ ] Reproducibility information
- [ ] No fabricated benchmark values
